In [1]:
import gc

In [2]:
from pathlib import Path

import numpy as np
import xarray as xr
from numpy import array, nan, inf, float32  # noqa: F401

In [3]:
from data import (
    ds_single,
    ds_pressure,
    _load_single_dataset,
    _load_pressure_dataset,
)

In [4]:
from dataclasses import dataclass


@dataclass
class EnsembleSpreadBounds:
    percentile: list[float]
    absolute: list[float]
    relative: list[float]
    quadratic: list[float]


@dataclass
class CompressionRatios:
    percentile: list[float]
    linquant: list[float]
    bitround: list[float]
    linquantquadstep: list[float]

In [5]:
def parse_single(path: Path):
    with Path(path).open() as f:
        for line in f.readlines():
            var, rest = line.split(": ", 1)
            info = eval(rest)

            yield var, info

In [6]:
def parse_pressure(path: Path):
    with Path(path).open() as f:
        ls = iter(f.readlines())

        line = None

        while True:
            if line is None:
                try:
                    line = next(ls)
                except Exception:
                    break

            var, rest = line.split(":", 1)

            while True:
                try:
                    line = next(ls)
                except Exception:
                    line = None
                    break

                if len(line.strip()) == 0:
                    continue

                if not line.startswith(" - "):
                    break

                level, rest = line[len(" - ") :].split(": ", 1)
                info = eval(rest)

                yield var, level, info

In [7]:
single_ensemble_spread_bounds = dict(parse_single("ensemble-spread-single.txt"))

In [8]:
pressure_ensemble_spread_bounds = dict()
for var, level, info in parse_pressure("ensemble-spread-pressure.txt"):
    if var not in pressure_ensemble_spread_bounds:
        pressure_ensemble_spread_bounds[var] = dict()
    pressure_ensemble_spread_bounds[var][level] = info

In [9]:
from numcodecs_wasm_bit_round import BitRound
from numcodecs_wasm_zstd import Zstd

In [10]:
lossless = Zstd(level=3)

In [11]:
def compute_absolute_error_compression_ratio(mean: np.ndarray, eb_abs: float) -> float:
    quantized = np.round(mean / eb_abs).astype(int)
    compressed = lossless.encode(quantized - np.amin(quantized))

    return mean.nbytes / compressed.nbytes


def compute_relative_error_compression_ratio(mean: np.ndarray, eb_rel: float) -> float:
    info = np.finfo(mean.dtype)

    keepbits_mantissa = np.ceil(-np.log2(eb_rel * 2.0))
    keepbits_mantissa = int(max(0, min(keepbits_mantissa, info.bits - info.iexp - 1)))

    rounded = BitRound(keepbits=keepbits_mantissa).encode(mean)
    compressed = lossless.encode(rounded)

    return mean.nbytes / compressed.nbytes


def compute_quadratic_error_compression_ratio(
    mean: np.ndarray, eb_qua: float, nbins=100
) -> float:
    amin, amax = np.amin(mean), np.amax(mean)
    ibin = np.minimum(
        np.round(nbins * (mean - amin) / (amax - amin)).astype(int), nbins - 1
    )

    eb_abs = eb_qua - eb_qua * (-1.0 + 2.0 * (ibin + 0.5) / (nbins))

    quantized = np.round(mean / eb_abs).astype(int)
    compressed = lossless.encode(quantized - np.amin(quantized))
    table = lossless.encode(ibin)

    return mean.nbytes / (compressed.nbytes + table.nbytes)


def compute_compression_ratios(
    mean: xr.DataArray, bounds: EnsembleSpreadBounds
) -> CompressionRatios:
    mean = mean.copy(deep=True).values

    linquant = []
    for p, eb_abs in zip(bounds.percentile, bounds.absolute):
        if not np.isfinite(eb_abs):
            linquant.append(np.nan)
        else:
            linquant.append(compute_absolute_error_compression_ratio(mean, eb_abs))

    bitround = []
    for p, eb_rel in zip(bounds.percentile, bounds.relative):
        if not np.isfinite(eb_rel):
            bitround.append(np.nan)
        else:
            bitround.append(compute_relative_error_compression_ratio(mean, eb_rel))

    linquantquadstep = []
    for p, eb_qua in zip(bounds.percentile, bounds.quadratic):
        if not np.isfinite(eb_qua):
            linquantquadstep.append(np.nan)
        else:
            linquantquadstep.append(
                compute_quadratic_error_compression_ratio(mean, eb_qua)
            )

    return CompressionRatios(
        percentile=bounds.percentile,
        linquant=linquant,
        bitround=bitround,
        linquantquadstep=linquantquadstep,
    )

In [12]:
from tqdm.notebook import tqdm

In [13]:
with open("compression-ratio-single.txt", "w") as f:
    for i, key in tqdm(enumerate(sorted(ds_single.mean)), position=0):
        if (i > 0) and (i % 50 == 0):
            del ds_single
            gc.collect()
            ds_single = _load_single_dataset()
        print(f"{key}: ", end="", flush=True, file=f)
        print(
            f"{compute_compression_ratios(ds_single.mean[key], single_ensemble_spread_bounds[key])!r}",
            flush=True,
            file=f,
        )

 80%|########  | 181/226 [00:00<?, ?it/s]

/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_21532/3274943806.py:2: RuntimeWarning: invalid value encountered in cast
  quantized = np.round(mean / eb_abs).astype(int)
/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_21532/3274943806.py:20: RuntimeWarning: invalid value encountered in cast
  ibin = np.minimum(np.round(nbins * (mean - amin) / (amax - amin)).astype(int), nbins-1)
/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_21532/3274943806.py:24: RuntimeWarning: invalid value encountered in cast
  quantized = np.round(mean / eb_abs).astype(int)
/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_21532/3274943806.py:2: RuntimeWarning: divide by zero encountered in divide
  quantized = np.round(mean / eb_abs).astype(int)
/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_21532/3274943806.py:2: RuntimeWarning: invalid value encountered in divide
  quantized = np.round(mean / eb_abs).astype(int)
/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7

In [14]:
with open("compression-ratio-pressure.txt", "w") as f:
    for i, key in tqdm(enumerate(sorted(ds_pressure.mean)), position=0):
        if (i > 0) and (i % 8 == 0):
            del ds_pressure
            gc.collect()
            ds_pressure = _load_pressure_dataset()
        print(f"{key}:", flush=True, file=f)
        for pressure in sorted(ds_pressure.mean[key].pressure_level.values):
            print(f" - {pressure} hPa: ", end="", flush=True, file=f)
            print(
                f"{compute_compression_ratios(ds_pressure.mean[key].sel(pressure_level=pressure), pressure_ensemble_spread_bounds[key][f'{pressure} hPa'])!r}",
                flush=True,
                file=f,
            )

0it [00:00, ?it/s]

/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_21532/3274943806.py:2: RuntimeWarning: invalid value encountered in divide
  quantized = np.round(mean / eb_abs).astype(int)
/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_21532/3274943806.py:2: RuntimeWarning: invalid value encountered in cast
  quantized = np.round(mean / eb_abs).astype(int)
/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_21532/3274943806.py:10: RuntimeWarning: divide by zero encountered in log2
  keepbits_mantissa = np.ceil(-np.log2(eb_rel * 2.0))
/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_21532/3274943806.py:20: RuntimeWarning: invalid value encountered in divide
  ibin = np.minimum(np.round(nbins * (mean - amin) / (amax - amin)).astype(int), nbins-1)
/var/folders/8v/swxsmn0d4vz5yzwjhf6bc26x3g7lq6/T/ipykernel_21532/3274943806.py:20: RuntimeWarning: invalid value encountered in cast
  ibin = np.minimum(np.round(nbins * (mean - amin) / (amax - amin)).astype(int), nbins-1